# Longitudinal Panel Data: Quality Control and Exploratory Analysis

This notebook demonstrates a data quality control (QC) and exploratory data analysis (EDA) pipeline for a four-wave longitudinal panel (waves `T1`–`T4`). The substantive question is how five personality traits and a behavioural exposure (weekly hours of video gaming) relate over time, and whether associations reflect **selection** (traits predict later exposure) or **socialisation** (exposure predicts later trait change). This is estimated with a Random Intercept Cross-Lagged Panel Model (RI-CLPM) and a Latent Growth Curve Model (LGCM).

**All data in this notebook are simulated.** Variable names, codings, sample size, wave structure and effect sizes are invented for demonstration purposes and do not correspond to any real dataset.

The pipeline covers:

1. **Data preparation**: simulation of a wide-format panel, recoding of categorical responses into numeric values, and construction of an exposure index.
2. **Missingness**: amount of missing data, retention across waves, and tests of the missingness mechanism (MCAR / MAR).
3. **Outliers**: cross-sectional and change (longitudinal) outliers, and their characteristics.
4. **Within-person variability**: spaghetti plots, Reliable Change Index and effect sizes of change.
5. **Measurement agreement**: agreement between two reports of the same exposure.
6. **Sample description**: descriptive statistics, correlations, group comparisons and exclusion criteria.

## 1. Data preparation

### Imports

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pandas.plotting import scatter_matrix
from pyampute.exploration.mcar_statistical_tests import MCARTest
from scipy.stats import chi2_contingency, pearsonr, spearmanr, ttest_ind
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
rng = np.random.default_rng(2024)

### Simulated panel

Each trait is generated as a stable person-level intercept plus an autoregressive within-person component. The exposure is a two-part process: a binary *user* indicator and, for users, categorical intensity (hours/day) and frequency (days/week) responses. Missingness is introduced with a mild dependence on previous scores (MAR), and a handful of implausible values are injected so that the QC steps have something to find.

In [ ]:
N_PERSONS = 1000
WAVES = ["T1", "T2", "T3", "T4"]
TRAITS = ["N", "E", "O", "A", "C"]


def simulate_ar(n, n_waves, ri_sd=0.5, phi=0.4, w_sd=0.3, mean=2.0):
    ri = rng.normal(0, ri_sd, n)
    w = np.zeros((n, n_waves))
    w[:, 0] = rng.normal(0, w_sd, n)
    for t in range(1, n_waves):
        w[:, t] = phi * w[:, t - 1] + rng.normal(0, w_sd * np.sqrt(1 - phi**2), n)
    return mean + ri[:, None] + w


traits_df = pd.DataFrame({"pid": np.arange(1, N_PERSONS + 1)})
traits_df["sex"] = rng.integers(0, 2, N_PERSONS)
base_age = rng.uniform(20, 30, N_PERSONS)
for k, wave in enumerate(WAVES):
    traits_df[f"age_{wave}"] = base_age + k + rng.normal(0, 0.1, N_PERSONS)

for trait in TRAITS:
    scores = simulate_ar(N_PERSONS, len(WAVES), mean=rng.uniform(1.8, 2.6))
    for k, wave in enumerate(WAVES):
        traits_df[f"{trait}_{wave}"] = np.clip(scores[:, k], 0, 4)

# Two-part exposure: user indicator + categorical intensity/frequency
exposure_df = pd.DataFrame({"pid": traits_df["pid"]})
latent_use = simulate_ar(N_PERSONS, len(WAVES), ri_sd=1.0, phi=0.5, w_sd=0.7, mean=0.0)
for k, wave in enumerate(WAVES):
    user = (latent_use[:, k] + 0.4 * traits_df["sex"] > -0.3).astype(int)
    exposure_df[f"user_{wave}"] = user
    exposure_df[f"int_cat_{wave}"] = np.where(user == 1, np.clip(np.round(2.5 + latent_use[:, k]), 1, 5), np.nan)
    exposure_df[f"freq_cat_{wave}"] = np.where(user == 1, np.clip(np.round(3 + latent_use[:, k]), 1, 5), np.nan)

# Problematic-use screening items (0-4), only answered by users at T3 and T4
screen_df = pd.DataFrame({"pid": traits_df["pid"]})
for wave in ["T3", "T4"]:
    users = exposure_df[f"user_{wave}"] == 1
    for item in range(1, 7):
        vals = np.clip(np.round(rng.gamma(1.2, 0.6, N_PERSONS)), 0, 4)
        screen_df[f"pu{item}_{wave}"] = np.where(users, vals, np.nan)
    screen_df[f"user_{wave}"] = exposure_df[f"user_{wave}"]

# MAR attrition: missingness at wave t depends on the trait score at t-1
for k in range(1, len(WAVES)):
    p_miss = 0.05 + 0.04 * k + 0.05 * (traits_df[f"N_{WAVES[k-1]}"] > 2.5)
    drop = rng.random(N_PERSONS) < p_miss
    wave_cols = [f"{t}_{WAVES[k]}" for t in TRAITS]
    traits_df.loc[drop, wave_cols] = np.nan
    exp_cols = [c for c in exposure_df.columns if c.endswith(WAVES[k])]
    exposure_df.loc[drop, exp_cols] = np.nan

# Inject a few implausible values
bad = rng.choice(N_PERSONS, 8, replace=False)
traits_df.loc[bad, "C_T2"] = rng.choice([0.0, 4.0], 8)

traits_df.shape, exposure_df.shape, screen_df.shape

### Variable transformations

Intensity and frequency are recorded as ordered categories. They are converted to numeric values using the midpoint of each category (e.g. a "1–2 h" category becomes 1.5 h). The exposure index is the product of intensity (hours/day) and frequency (days/week), i.e. **hours per week**. Non-users are coded as zero exposure instead of missing.

In [ ]:
# Category code -> midpoint (illustrative codings)
intensity_map = {1: 0.5, 2: 1.5, 3: 3.0, 4: 5.0, 5: 8.0}   # hours per day
frequency_map = {1: 0.5, 2: 1.5, 3: 3.5, 4: 5.5, 5: 7.0}   # days per week

for wave in WAVES:
    exposure_df[f"hours_{wave}"] = exposure_df[f"int_cat_{wave}"].map(intensity_map)
    exposure_df[f"days_{wave}"] = exposure_df[f"freq_cat_{wave}"].map(frequency_map)
    exposure_df[f"exp_{wave}"] = exposure_df[f"hours_{wave}"] * exposure_df[f"days_{wave}"]

    non_user = exposure_df[f"user_{wave}"] == 0
    exposure_df.loc[non_user, [f"hours_{wave}", f"days_{wave}", f"exp_{wave}"]] = 0

EXP_COLS = [f"exp_{w}" for w in WAVES]
exposure_df[EXP_COLS].describe()

The raw exposure index shows a strong floor effect:

In [ ]:
def hist_grid(df, cols, xlabel="Value", bins=30):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for ax, col in zip(axes.flatten(), cols):
        ax.hist(df[col].dropna(), bins=bins)
        ax.set_title(col)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()


hist_grid(exposure_df, EXP_COLS)

We therefore treat it as a two-part variable. The binary part (`ever_user`) separates people who reported any exposure during the study from those who never did; the continuous part captures the amount of exposure among users:

In [ ]:
traits_df["ever_user"] = exposure_df[EXP_COLS].fillna(0).ne(0).any(axis=1).astype(int)
traits_df["ever_user"].value_counts()

In [ ]:
users_df = exposure_df[traits_df["ever_user"] == 1].copy()
hist_grid(users_df, EXP_COLS)

A log transformation (robust to zeros) substantially reduces the skew:

In [ ]:
LOG_COLS = [c + "_log" for c in EXP_COLS]
for col in EXP_COLS:
    users_df[col + "_log"] = np.log1p(users_df[col])

hist_grid(users_df, LOG_COLS, xlabel="log(1 + value)")

The variable is still zero-inflated but much closer to a Gaussian shape, which helps model estimation.

## 2. Missingness

A systematic evaluation of missingness is needed to justify the estimation strategy. Mean imputation and similar ad hoc methods (e.g. last observation carried forward) are incompatible with an RI-CLPM: they shrink within-person variance, which is exactly the signal the model separates from stable between-person differences, and they bias autoregressive and cross-lagged paths toward zero.

- **MCAR**: missingness unrelated to any observed or unobserved data. Even listwise deletion is unbiased (but inefficient).
- **MAR**: missingness may depend on observed variables. Full Information Maximum Likelihood (FIML) is unbiased under this assumption.
- **MNAR**: missingness depends on the unobserved values themselves; the missingness process must then be modelled explicitly.

FIML estimates the parameters directly from all available data points without imputation, preserving the covariance structure and the within-person dynamics.

### Missing values

In [ ]:
def missing_summary(df, cols=None, mask=None):
    sub = df if mask is None else df[mask]
    sub = sub if cols is None else sub[cols]
    return pd.DataFrame({
        "missing_count": sub.isna().sum(),
        "missing_percent": sub.isna().mean() * 100,
    }).sort_values("missing_percent", ascending=False)


TRAIT_COLS = [f"{t}_{w}" for t in TRAITS for w in WAVES]
missing_summary(traits_df, TRAIT_COLS)

In [ ]:
# Screening items: only evaluated among those eligible at each wave
for wave in ["T3", "T4"]:
    cols = [c for c in screen_df.columns if c.startswith("pu") and c.endswith(wave)]
    print(f"\n=== {wave} (eligible only) ===")
    print(missing_summary(screen_df, cols, mask=screen_df[f"user_{wave}"] == 1))

### Retention

High retention preserves sample comparability across waves; differential dropout threatens both representativeness and power. Pairwise coverage shows the proportion of participants with data at both waves:

In [ ]:
def coverage_matrix(df, var):
    cols = [f"{var}_{w}" for w in WAVES]
    cov = pd.DataFrame(index=WAVES, columns=WAVES, dtype=float)
    for (wi, ci), (wj, cj) in itertools.product(zip(WAVES, cols), repeat=2):
        cov.loc[wi, wj] = df[[ci, cj]].dropna().shape[0] / len(df)
    return cov.round(3)


def complete_case_pct(df, var):
    return df[[f"{var}_{w}" for w in WAVES]].notna().all(axis=1).mean() * 100


for var in TRAITS:
    print(f"\n=== {var} ===")
    print(coverage_matrix(traits_df, var))
    print(f"Complete cases (all four waves): {complete_case_pct(traits_df, var):.2f}%")

In [ ]:
print(coverage_matrix(exposure_df, "exp"))
print(f"Complete cases (all four waves): {complete_case_pct(exposure_df, 'exp'):.2f}%")

### Missingness mechanism

Little's MCAR test is run for each variable. When MCAR is rejected, the mechanism is conservatively treated as MAR, under which FIML remains appropriate.

In [ ]:
def mcar_test(df, var):
    cols = [f"{var}_{w}" for w in WAVES]
    result = MCARTest(method="little")(df[cols])
    print(f"Little's MCAR test, {var}: p = {result:.4f}")


for var in TRAITS:
    mcar_test(traits_df, var)
mcar_test(exposure_df, "exp")

To probe a MAR pattern, missingness at each wave is regressed on the observed score at the previous wave:

In [ ]:
def mar_check(df, var):
    rows = []
    for prev, curr in zip(WAVES[:-1], WAVES[1:]):
        y = df[f"{var}_{curr}"].isna().astype(int)
        X = df[[f"{var}_{prev}"]]
        valid = X.notna().all(axis=1)
        model = sm.Logit(y[valid], sm.add_constant(X[valid])).fit(disp=0)
        rows.append({
            "predictor": f"{var}_{prev}",
            "outcome": f"missing {var}_{curr}",
            "OR": np.exp(model.params.iloc[1]),
            "p": model.pvalues.iloc[1],
        })
    return pd.DataFrame(rows)


pd.concat([mar_check(traits_df, v) for v in TRAITS], ignore_index=True).round(4)

## 3. Outliers

Atypical observations can drive parameter estimates and distort distributions. **Cross-sectional outliers** are extreme values within a wave; **change outliers** are implausibly large jumps in an individual's trajectory. Traits are expected to be fairly stable, so abrupt changes are especially suspicious. Values beyond ±3 SD are flagged and inspected, not excluded automatically.

### Cross-sectional outliers

In [ ]:
def flag_outliers(df, cols, k=3, center="mean"):
    mask = pd.DataFrame(False, index=df.index, columns=cols)
    for col in cols:
        c = df[col].mean() if center == "mean" else df[col].median()
        sd = df[col].std()
        mask[col] = (df[col] < c - k * sd) | (df[col] > c + k * sd)
    return mask


trait_out = flag_outliers(traits_df, TRAIT_COLS)
traits_df["outlier"] = trait_out.any(axis=1).astype(int)

print("Rows with at least one outlier:", traits_df["outlier"].sum())
trait_out.sum().sort_values(ascending=False).head(10)

In [ ]:
traits_df[TRAIT_COLS].boxplot(figsize=(12, 6), rot=90)
plt.title("Trait scores by wave")
plt.show()

In [ ]:
exp_out = flag_outliers(exposure_df, EXP_COLS, center="median")
print("Rows with at least one exposure outlier:", exp_out.any(axis=1).sum())
exp_out.sum()

### Change outliers

In [ ]:
change_df = pd.DataFrame(index=traits_df.index)
for var in TRAITS:
    for prev, curr in zip(WAVES[:-1], WAVES[1:]):
        change_df[f"{var}_{prev}_{curr}"] = traits_df[f"{var}_{curr}"] - traits_df[f"{var}_{prev}"]

change_out = flag_outliers(change_df, change_df.columns)
summary = pd.DataFrame({
    "lower": [(change_df[c] < change_df[c].mean() - 3 * change_df[c].std()).sum() for c in change_df],
    "upper": [(change_df[c] > change_df[c].mean() + 3 * change_df[c].std()).sum() for c in change_df],
}, index=change_df.columns)
summary["total"] = summary.sum(axis=1)

print(summary.sort_values("total", ascending=False))
print("\nParticipants with at least one change outlier:", change_out.any(axis=1).sum())

Change outliers in the exposure are not treated as errors, since abrupt behavioural changes are plausible.

### Outlier characteristics

Do trait outliers differ in exposure status? A contingency table, a chi-squared test of independence and a logistic regression are used:

In [ ]:
ct = pd.crosstab(traits_df["outlier"], traits_df["ever_user"])
print(ct)
print(ct.div(ct.sum(axis=1), axis=0).round(3))

chi2, p, dof, _ = chi2_contingency(ct)
print(f"\nChi2 = {chi2:.3f}, df = {dof}, p = {p:.4f}")

logit = sm.Logit(traits_df["ever_user"], sm.add_constant(traits_df["outlier"])).fit(disp=0)
print(logit.summary())

Next, outliers are checked for **directional consistency**: a person who is an extreme case at one wave should stay on the same side of the mean at the other waves. Sign flips suggest measurement error.

In [ ]:
z_df = (traits_df[TRAIT_COLS] - traits_df[TRAIT_COLS].mean()) / traits_df[TRAIT_COLS].std()
z_out = z_df.abs() > 3


def is_consistent(row_z, row_out):
    extreme = row_z[row_out].dropna()
    if extreme.empty:
        return pd.NA
    if not ((extreme > 0).all() or (extreme < 0).all()):
        return False
    sign = np.sign(extreme.iloc[0])
    others = row_z[~row_out].dropna()
    return bool((np.sign(others) == sign).all()) if len(others) else True


consistency = pd.Series(pd.NA, index=traits_df.index, dtype="boolean")
for var in TRAITS:
    cols = [f"{var}_{w}" for w in WAVES]
    for i in z_df.index[z_out[cols].any(axis=1)]:
        res = is_consistent(z_df.loc[i, cols], z_out.loc[i, cols])
        if pd.isna(res):
            continue
        consistency[i] = res if pd.isna(consistency[i]) else (consistency[i] and res)

traits_df["outlier_consistent"] = consistency
traits_df["outlier_consistent"].value_counts(dropna=False)

In [ ]:
for i in traits_df.index[traits_df["outlier_consistent"] == False]:
    for var in TRAITS:
        cols = [f"{var}_{w}" for w in WAVES]
        if z_out.loc[i, cols].any():
            print(f"\nRow {i}, trait {var}")
            print(pd.DataFrame({"z": z_df.loc[i, cols].round(2), "outlier": z_out.loc[i, cols]}))

## 4. Within-person variability

Within-person models rely on fluctuations around each person's own mean. A variable with little within-person variation carries almost no information about within-person effects, so variability is checked before modelling.

### Change plots

In [ ]:
def spaghetti_plot(ax, df, var, max_lines=300):
    data = df[[f"{var}_{w}" for w in WAVES]].copy()
    data.columns = WAVES
    sample = data.sample(min(max_lines, len(data)), random_state=1)
    for _, row in sample.iterrows():
        ax.plot(WAVES, row.values, alpha=0.15, linewidth=0.8)
    ax.plot(WAVES, data.mean().values, color="red", linewidth=2.5, label="Mean")
    ax.set_title(var)
    ax.set_xlabel("Wave")


fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, var in zip(axes.flatten(), TRAITS):
    spaghetti_plot(ax, traits_df, var)
spaghetti_plot(axes.flatten()[-1], exposure_df, "exp")
axes.flatten()[0].legend()
plt.tight_layout()
plt.show()

### Reliable Change Index

The Reliable Change Index (RCI) tests whether an individual's change exceeds what would be expected from measurement error, given the reliability of the scale. The reliability values below are **placeholders** standing in for published test–retest coefficients. The RCI is not computed for the exposure, since it is a behaviour rather than a trait and test–retest reliability does not apply.

In [ ]:
def rci_change(df, var, reliability, threshold=1.96):
    cols = {w: f"{var}_{w}" for w in WAVES}
    se_diff = df[cols[WAVES[0]]].std() * np.sqrt(2 * (1 - reliability))
    any_change = pd.Series(False, index=df.index)
    rows = []
    for t1, t2 in itertools.combinations(WAVES, 2):
        changed = ((df[cols[t2]] - df[cols[t1]]) / se_diff).abs().gt(threshold).fillna(False)
        any_change |= changed
        rows.append({"comparison": f"{t1}-{t2}", "n_changed": changed.sum(), "percent": 100 * changed.mean()})
    return any_change.sum(), 100 * any_change.mean(), pd.DataFrame(rows)


reliability = {"N": 0.85, "E": 0.85, "O": 0.85, "A": 0.85, "C": 0.85}  # placeholder values

for var in TRAITS:
    n, pct, table = rci_change(traits_df, var, reliability[var])
    print(f"\n=== {var} ===  {n} participants ({pct:.2f}%) changed reliably at least once")
    print(table.round(2).to_string(index=False))

### Effect sizes

Cohen's *d* for paired change complements the RCI by quantifying the standardised magnitude of change between each pair of waves:

In [ ]:
def cohens_d_pairs(df, var):
    rows = []
    for t1, t2 in itertools.combinations(WAVES, 2):
        diff = (df[f"{var}_{t2}"] - df[f"{var}_{t1}"]).dropna()
        rows.append({"comparison": f"{t1}-{t2}", "variable": var, "d": diff.mean() / diff.std()})
    return pd.DataFrame(rows)


d_table = pd.concat([cohens_d_pairs(traits_df, v) for v in TRAITS] + [cohens_d_pairs(exposure_df, "exp")])
d_table.pivot(index="comparison", columns="variable", values="d").round(3)

## 5. Measurement agreement

When the same exposure is reported twice (for example, a concurrent report and a later recalled report of the same period), their agreement gives an estimate of reporting bias. Here a second, noisier report of the `T2` exposure is simulated and compared with the original.

In [ ]:
exposure_df["exp_T2_alt"] = np.clip(
    exposure_df["exp_T2"] * rng.normal(1.1, 0.35, N_PERSONS) + rng.normal(0, 2, N_PERSONS), 0, None
)

rho, p = spearmanr(exposure_df["exp_T2"], exposure_df["exp_T2_alt"], nan_policy="omit")
print(f"Spearman rho = {rho:.3f}, p = {p:.3g}")

bias = exposure_df["exp_T2_alt"] - exposure_df["exp_T2"]
print(bias.describe())

In [ ]:
q1, med, q3 = bias.quantile([0.25, 0.5, 0.75])
plt.figure(figsize=(8, 5))
plt.hist(bias.dropna(), bins=30)
plt.axvline(0, linestyle="--", color="k", label="No bias")
plt.axvline(med, label=f"Median = {med:.2f}")
plt.axvline(q1, linestyle=":", label=f"Q1 = {q1:.2f}")
plt.axvline(q3, linestyle=":", label=f"Q3 = {q3:.2f}")
plt.xlabel("Second report - first report (hours/week)")
plt.ylabel("Frequency")
plt.title("Distribution of reporting differences")
plt.legend()
plt.show()

## 6. Sample description

### Descriptive statistics

In [ ]:
traits_df[TRAIT_COLS].describe().T.round(2)

In [ ]:
exposure_df[EXP_COLS].describe().T.round(2)

### Correlations

Person-level means across waves are inspected first, to check whether any traits are redundant:

In [ ]:
mean_df = pd.DataFrame({v: traits_df[[f"{v}_{w}" for w in WAVES]].mean(axis=1) for v in TRAITS})
mean_df["exp"] = exposure_df[EXP_COLS].mean(axis=1)

scatter_matrix(mean_df, figsize=(10, 10), alpha=0.3)
plt.suptitle("Scatterplot matrix of person-level means")
plt.show()

A helper builds a correlation table with FDR-corrected significance stars:

In [ ]:
def corr_with_significance(df):
    cols = df.columns
    r = pd.DataFrame(index=cols, columns=cols, dtype=float)
    p = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for c1, c2 in itertools.combinations(cols, 2):
        d = df[[c1, c2]].dropna()
        if len(d) > 2:
            r.loc[c1, c2], p.loc[c1, c2] = pearsonr(d[c1], d[c2])
            r.loc[c2, c1], p.loc[c2, c1] = r.loc[c1, c2], p.loc[c1, c2]

    iu = np.triu_indices(len(cols), k=1)
    p_arr = p.to_numpy(copy=True)
    adj = multipletests(p_arr[iu], method="fdr_bh")[1]
    p_arr[iu] = adj
    p_arr[(iu[1], iu[0])] = adj
    p_adj = pd.DataFrame(p_arr, index=cols, columns=cols)

    def fmt(rv, pv):
        stars = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else ""
        return f"{rv:.2f}{stars}"

    return pd.DataFrame(
        [[fmt(r.iloc[i, j], p_adj.iloc[i, j]) if i != j else "1" for j in range(len(cols))] for i in range(len(cols))],
        index=cols, columns=cols,
    )


corr_with_significance(mean_df)

Correlations of change scores (within-person associations) for each adjacent pair of waves and for first-to-last change:

In [ ]:
pairs = list(zip(WAVES[:-1], WAVES[1:])) + [(WAVES[0], WAVES[-1])]
for t1, t2 in pairs:
    delta = pd.DataFrame({v: traits_df[f"{v}_{t2}"] - traits_df[f"{v}_{t1}"] for v in TRAITS})
    delta["exp"] = exposure_df[f"exp_{t2}"] - exposure_df[f"exp_{t1}"]
    print(f"\n=== Change {t1} -> {t2} ===")
    print(corr_with_significance(delta))

### Group comparisons

Welch t-tests with Cohen's *d* and FDR correction, used both for users vs non-users at each wave and for sex differences (a well-known pattern, used here as a sanity check of the data).

In [ ]:
def group_compare(df, cols, group, labels=("group_0", "group_1")):
    rows = []
    for col in cols:
        g0 = df.loc[df[group] == 0, col].dropna()
        g1 = df.loc[df[group] == 1, col].dropna()
        t, p = ttest_ind(g0, g1, equal_var=False)
        d = (g1.mean() - g0.mean()) / np.sqrt((g0.var() + g1.var()) / 2)
        rows.append({"variable": col, f"mean_{labels[0]}": g0.mean(), f"mean_{labels[1]}": g1.mean(),
                     "t": t, "p": p, "d": d})
    out = pd.DataFrame(rows)
    out["p_adj"] = multipletests(out["p"], method="fdr_bh")[1]
    return out.round(3)


for wave in WAVES:
    tmp = traits_df.assign(user=exposure_df[f"user_{wave}"])
    print(f"\n=== Users vs non-users, {wave} ===")
    print(group_compare(tmp, [f"{v}_{wave}" for v in TRAITS], "user", ("non_users", "users")))

In [ ]:
for wave in [WAVES[0], WAVES[-1]]:
    print(f"\n=== Sex differences, {wave} ===")
    print(group_compare(traits_df, [f"{v}_{wave}" for v in TRAITS], "sex", ("sex0", "sex1")))

print("\n=== Sex differences in change scores ===")
print(group_compare(change_df.assign(sex=traits_df["sex"]), list(change_df.columns), "sex", ("sex0", "sex1")))

### Users vs non-users controlling for sex

Logistic regressions test whether each trait (person-level mean) predicts being a user, adjusting for sex:

In [ ]:
reg_df = mean_df.assign(ever_user=traits_df["ever_user"], sex=traits_df["sex"]).dropna()
rows = []
for var in TRAITS:
    m = sm.Logit(reg_df["ever_user"], sm.add_constant(reg_df[[var, "sex"]])).fit(disp=0)
    rows.append({"trait": var, "OR": np.exp(m.params[var]), "p": m.pvalues[var]})
pd.DataFrame(rows).round(4)

Model-based, sex-adjusted trajectories by group (`score ~ ever_user * wave + sex`) with 95% confidence bands:

In [ ]:
long_df = (
    traits_df[["pid", "sex", "ever_user"] + TRAIT_COLS]
    .melt(id_vars=["pid", "sex", "ever_user"], var_name="var", value_name="score")
    .dropna()
)
long_df[["trait", "wave"]] = long_df["var"].str.split("_", expand=True)

preds = []
for var in TRAITS:
    model = smf.ols("score ~ ever_user * C(wave) + sex", data=long_df[long_df["trait"] == var]).fit()
    grid = pd.DataFrame({"ever_user": [0] * 4 + [1] * 4, "wave": WAVES * 2, "sex": long_df["sex"].mean()})
    frame = model.get_prediction(grid).summary_frame(alpha=0.05)
    grid["pred"], grid["lo"], grid["hi"] = frame["mean"], frame["mean_ci_lower"], frame["mean_ci_upper"]
    grid["trait"] = var
    grid["group"] = grid["ever_user"].map({0: "Non-users", 1: "Users"})
    preds.append(grid)
pred_df = pd.concat(preds)

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
for ax, var in zip(axes.flatten(), TRAITS):
    for group, d in pred_df[pred_df["trait"] == var].groupby("group"):
        ax.plot(WAVES, d["pred"], marker="o", label=group)
        ax.fill_between(WAVES, d["lo"], d["hi"], alpha=0.2)
    ax.set_title(var)
axes.flatten()[-1].axis("off")
axes.flatten()[0].legend()
fig.supylabel("Sex-adjusted score (model-based)")
plt.tight_layout()
plt.show()

### Exclusion criteria

Participants with an unusually high problematic-use screening score may follow a different developmental pattern, so they are flagged for exclusion. Rather than a diagnostic cut-off, a statistical rule is used: a sum score more than 2.5 SD above the mean among users at either wave in which the screening was administered.

In [ ]:
flags = pd.Series(False, index=screen_df.index)
for wave in ["T3", "T4"]:
    items = [c for c in screen_df.columns if c.startswith("pu") and c.endswith(wave)]
    eligible = screen_df[f"user_{wave}"] == 1
    score = screen_df.loc[eligible, items].sum(axis=1, min_count=1)
    flags.loc[score.index] |= score > score.mean() + 2.5 * score.std()

traits_df["exclude"] = flags.astype(int)
print("Participants flagged for exclusion:", traits_df["exclude"].sum())

### Export for Mplus

The cleaned analytic sample is written out in the format expected by the Mplus input files (`RICLPM.inp`, `LGCM.inp`).

In [ ]:
analytic = traits_df[traits_df["exclude"] == 0].merge(exposure_df[["pid"] + EXP_COLS], on="pid")

ri_df = pd.DataFrame({"pid": analytic["pid"]})
for k, w in enumerate(WAVES, start=1):
    ri_df[f"x{k}"] = analytic[f"N_{w}"]
    ri_df[f"y{k}"] = np.log1p(analytic[f"exp_{w}"])
ri_df.to_csv("riclpm_data.csv", index=False, header=False, na_rep=".")

lg_df = pd.DataFrame({"pid": analytic["pid"]})
for k, w in enumerate(WAVES, start=1):
    lg_df[f"x{k}"] = analytic[f"C_{w}"]
lg_df["grp"] = analytic["ever_user"]
lg_df["sex"] = analytic["sex"]
for k, w in enumerate(WAVES, start=1):
    lg_df[f"age{k}"] = analytic[f"age_{w}"]
lg_df.to_csv("lgcm_data.dat", index=False, header=False, sep=" ", na_rep=".")